# 03 - BioNeMo ESM-2 mutation embeddings

**The core idea.** Do not embed the whole mutant protein. DNMT3A is 912
residues; changing one of them barely moves a mean-pooled vector. Take the
representation *at the mutated residue* and subtract the wild type:

$$\Delta = \mathrm{ESM2}(s_{\text{mut}})[i] - \mathrm{ESM2}(s_{\text{wt}})[i]$$

**Runtime.** Two forward passes per unique mutation. Deduplicate first - R882H
appears in hundreds of patients but only needs embedding once. On a Colab T4,
650M BioNeMo ESM-2 handles roughly 3-5 windowed sequences/second.

**Iterate small.** Debug with `nvidia/esm2_t12_35M_UR50D`; report with `nvidia/esm2_t33_650M_UR50D`.
Falls back to the matching `facebook/` checkpoint if NVIDIA weights are unavailable.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    # Run every session, not only on first mount: Colab wipes packages on
    # restart, and misspelled package names fail silently without check=True.
    subprocess.run("pip install -q transformers accelerate lightgbm".split(), check=True)
    # Optional on A100/H100: pip install transformer-engine  (BioNeMo TE kernels)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

CFG = yaml.safe_load(open("configs/config.yaml"))
CFG

Mounted at /content/drive
working dir: /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection


{'project': {'name': 'blood-cancer-early-detection', 'seed': 42},
 'data': {'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'external_dir': 'data/external',
  'maf_file': 'data/raw/data_mutations_extended.txt',
  'clinical_file': 'data/raw/data_clinical_patient.txt',
  'uniprot_cache': 'data/external/uniprot_cache.json'},
 'filters': {'min_vaf': 0.02,
  'min_depth': 0,
  'keep_kinds': ['missense', 'nonsense', 'frameshift', 'splice', 'indel'],
  'use_ch_panel': True},
 'labels': {'strategy': 'longitudinal',
  'outcome_col': 'progressed_to_aml',
  'positive_diagnoses': ['AML', 'MDS', 'MPN', 'CMML']},
 'embedding': {'model_name': 'nvidia/esm2_t33_650M_UR50D',
  'batch_size': 1,
  'weight_by_vaf': True},
 'model': {'kind': 'logreg',
  'n_pca': 32,
  'n_splits': 5,
  'target_specificity': 0.98,
  'out_path': 'models/classifier.joblib'}}

In [3]:
import json
from pathlib import Path
from src.embedding import ESM2Embedder, window_sequence, apply_missense

processed = Path(CFG["data"]["processed_dir"])
cache_path = Path(CFG["data"]["uniprot_cache"])
if not cache_path.exists():
    cache_path = Path("data/external/uniprot_cache.json")

targets = pd.read_csv(processed / "lbc_sardinia_unique_targets.csv")
print("targets", targets.shape, targets.columns.tolist())

raw_cache = json.loads(cache_path.read_text())
# cache có thể là {gene: seq} hoặc {uniprot: {gene, seq}}
sequences = {}
for k, v in raw_cache.items():
    if isinstance(v, str) and v and v[0] in "ACDEFGHIKLMNPQRSTVWY":
        sequences[k] = v
    elif isinstance(v, dict):
        seq = v.get("seq") or v.get("sequence")
        gene = v.get("gene") or k
        if seq:
            sequences[str(gene)] = seq
print("sequences", len(sequences), "missing genes",
      sorted(set(targets.gene) - set(sequences)))

mut = targets.copy()
mut["kind"] = "missense"
labelled = None

targets (266, 11) ['mut_key', 'gene', 'protein_substitution', 'aa_wt', 'aa_pos', 'aa_mut', 's', 's_z', 'n_obs', 'n_rows', 'cohort']
sequences 62 missing genes []


In [4]:
MODEL = CFG.get("embedding", {}).get("model_name", "nvidia/esm2_t33_650M_UR50D")
if isinstance(MODEL, str) and MODEL.startswith("facebook/"):
    MODEL = "nvidia/" + MODEL.split("/", 1)[1]
# MODEL = "nvidia/esm2_t12_35M_UR50D"   # uncomment while iterating
print("embedding model:", MODEL)
embedder = ESM2Embedder(MODEL)
print("resolved:", embedder.model_name, embedder.backend)


embedding model: nvidia/esm2_t33_650M_UR50D
[bionemo-esm2] loading nvidia/esm2_t33_650M_UR50D on cuda torch.float16 ...


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

The repository nvidia/esm2_t33_650M_UR50D contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/nvidia/esm2_t33_650M_UR50D .
 You can inspect the repository content at https://hf.co/nvidia/esm2_t33_650M_UR50D.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


esm_nv.py:   0%|          | 0.00/38.6k [00:00<?, ?B/s]

[transformers] Encountered exception while importing transformer_engine: No module named 'transformer_engine'


[bionemo-esm2] nvidia/esm2_t33_650M_UR50D failed: This modeling file requires the following packages that were not found in your environment: transformer_engine. Run `pip install transformer_engine`
[bionemo-esm2] loading facebook/esm2_t33_650M_UR50D on cuda torch.float16 ...


... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[bionemo-esm2] ready  backend=facebook_hf  model=facebook/esm2_t33_650M_UR50D  hidden dim=1280  alloc=1302MB reserved=1428MB
resolved: facebook/esm2_t33_650M_UR50D facebook_hf


In [5]:
# Smoke test on one hotspot before committing to the full run.
from src.embedding import window_sequence, apply_missense

seq = sequences["DNMT3A"]
print("residue at 882:", seq[881], "(expect R on real UniProt data)")

mut_seq = apply_missense(seq, seq[881], 882, "H")
w, p = window_sequence(seq, 882)
m, _ = window_sequence(mut_seq, 882)

d = embedder.encode([m], positions=[p]) - embedder.encode([w], positions=[p])
print("delta shape:", d.shape, "| L2 norm:", float(np.linalg.norm(d)))

residue at 882: R (expect R on real UniProt data)
  [bionemo-esm2] 1/1  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 1/1  alloc=1312MB reserved=1493MB
delta shape: (1, 1280) | L2 norm: 2.849716901779175


In [6]:
%%time
emb_table = embed_mutations(mut, sequences, embedder,
                            batch_size=CFG["embedding"]["batch_size"])
emb_table.head()

[embed] 266 rows -> 266 unique missense mutations
[embed] mapped 260/266 (reference mismatches: 34, 13.1%)
  [bionemo-esm2] 1/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 21/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 41/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 61/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 81/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 101/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 121/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 141/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 161/260  alloc=1312MB reserved=1481MB
  [bionemo-esm2] 181/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 201/260  alloc=1312MB reserved=1508MB
  [bionemo-esm2] 221/260  alloc=1312MB reserved=1508MB
  [bionemo-esm2] 241/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 1/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 21/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 41/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 61/260

,mut_key,gene,aa_wt,aa_pos,aa_mut,delta,backend,model_name
0,ASXL1_M341L,ASXL1,M,341,L,"[-0.05154419, 0.107284546, 0.040691376, 0.0010...",facebook_hf,facebook/esm2_t33_650M_UR50D
1,ASXL1_Q803R,ASXL1,Q,803,R,"[0.07937622, 0.113220215, -0.024475098, -0.047...",facebook_hf,facebook/esm2_t33_650M_UR50D
2,BCORL1_D1154A,BCORL1,D,1154,A,"[0.01829338, 0.07763672, -0.0050964355, -0.122...",facebook_hf,facebook/esm2_t33_650M_UR50D
3,BCORL1_D1198A,BCORL1,D,1198,A,"[0.013946533, 0.17529297, 0.014631271, -0.1103...",facebook_hf,facebook/esm2_t33_650M_UR50D
4,BCORL1_D1514A,BCORL1,D,1514,A,"[-0.05634439, 0.11830139, 0.008132935, 0.25100...",facebook_hf,facebook/esm2_t33_650M_UR50D
